# conv-windowing-1d — worked example 2: Dilated 1-D conv windows via as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-1d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A **dilated** convolution spreads the kernel taps apart by inserting `D-1` gaps between them, so a kernel of width `KW` covers an effective span of `(KW - 1) * D + 1` input elements. With `as_strided` the gap is expressed by giving the `KW` axis a stride of `D * s_w` instead of `s_w`, while the window-start (`OW`) axis keeps stride `s_w` (stride-1 between windows).

## Worked solution

**Goal.** Build the `(B, IC, OW, KW)` view for a dilation-`D` conv and check it against `F.conv1d(..., dilation=D)`.

1. **Effective kernel span.** Dilation places taps at positions `0, D, 2D, ...`, so the kernel reaches `(KW - 1) * D + 1` elements. Call this `eff`.
2. **Output width.** The last window can start at `W - eff`, with starts spaced 1 apart (stride-1 between windows), so `OW = W - eff + 1`.
3. **Strides for the two new axes.** Adjacent windows are offset by one input element, so the `OW` axis stride is `s_w`. *Within* a window, consecutive taps are `D` input elements apart, so the `KW` axis stride is `D * s_w` — this is the only change from the plain stride-1 case.
4. **as_strided call.** `x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, s_w, D * s_w))`. Still a view.
5. **Why it matches.** Each gathered window now picks the dilated taps directly, so contracting with the kernel reproduces the dilated dot-products of `F.conv1d`.

In [ ]:
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_windows_dilated(x: t.Tensor, KW: int, D: int) -> t.Tensor:
    B, IC, W = x.shape
    eff = (KW - 1) * D + 1
    OW = W - eff + 1
    s_b, s_ic, s_w = x.stride()
    return x.as_strided(size=(B, IC, OW, KW), stride=(s_b, s_ic, s_w, D * s_w))

t.manual_seed(0)
x = t.randn(2, 3, 12)
weight = t.randn(5, 3, 3)
KW, D = 3, 2
win = conv1d_windows_dilated(x, KW, D)
out = einsum(win, weight, 'b i o k, c i k -> b c o')
ref = F.conv1d(x, weight, dilation=D)
print('window shape:', tuple(win.shape))
print('shares storage:', win.data_ptr() == x.data_ptr())
print('matches conv1d:', t.allclose(out, ref, atol=1e-4))